# 05.7 - Gradient Boosting

**Phase:** 05 - Machine Learning

**Status:** VERIFIED

---

## 1. What Are We Solving?

While random forests build trees **in parallel** (bagging), gradient boosting builds trees **sequentially**, each correcting the errors of the previous ones. This produces state-of-the-art tabular models.

## 2. Why Does This Matter?

Gradient boosting (XGBoost, LightGBM, CatBoost) wins most tabular ML competitions. Understanding boosting is essential for high-performance modeling.

## 3. Prerequisites

- Unit 05.5 (Decision Trees), Unit 05.6 (Random Forests)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Explain the boosting concept
- Explain gradient boosting
- Train and compare XGBoost and LightGBM
- Understand the difference from bagging

## 5. Mental Model

Boosting:

1. Train a weak model (shallow tree).
2. Compute residuals (errors).
3. Train the next tree to predict the residuals.
4. Add it with a small learning rate.
5. Repeat.

Final prediction = sum of all trees' contributions.

Bagging reduces variance; boosting reduces bias.


## 6. Generate Data

Use a classification dataset.


In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score
import xgboost as xgb
import lightgbm as lgb

np.random.seed(42)
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, n_redundant=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Train: {len(X_train)}, Test: {len(X_test)}")


Train: 700, Test: 300


## 7. Gradient Boosting (sklearn)

Train a gradient boosting classifier.


In [2]:
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
gb_acc = accuracy_score(y_test, gb.predict(X_test))
print(f"Gradient boosting test accuracy: {gb_acc:.3f}")


Gradient boosting test accuracy: 0.907


## 8. Compare: Random Forest vs Gradient Boosting

Both are tree ensembles but use different strategies.


In [3]:
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf.predict(X_test))
print(f"Random forest:      {rf_acc:.3f}")
print(f"Gradient boosting:  {gb_acc:.3f}")
print("\nBoosting often edges out forests on structured data.")


Random forest:      0.917
Gradient boosting:  0.907

Boosting often edges out forests on structured data.


## 9. XGBoost

XGBoost is a highly optimized gradient boosting library.


In [4]:
xgb_model = xgb.XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
xgb_acc = accuracy_score(y_test, xgb_model.predict(X_test))
print(f"XGBoost test accuracy: {xgb_acc:.3f}")


XGBoost test accuracy: 0.917


## 10. LightGBM

LightGBM uses histogram-based learning for speed.


In [5]:
lgb_model = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1)
lgb_model.fit(X_train, y_train)
lgb_acc = accuracy_score(y_test, lgb_model.predict(X_test))
print(f"LightGBM test accuracy: {lgb_acc:.3f}")


LightGBM test accuracy: 0.933


## 11. Effect of Learning Rate

Lower learning rate needs more trees but often generalizes better.


In [6]:
for lr in [0.01, 0.1, 0.3]:
    m = xgb.XGBClassifier(n_estimators=200, learning_rate=lr, random_state=42, eval_metric='logloss')
    m.fit(X_train, y_train)
    acc = accuracy_score(y_test, m.predict(X_test))
    print(f"learning_rate={lr}: test accuracy={acc:.3f}")


learning_rate=0.01: test accuracy=0.910


learning_rate=0.1: test accuracy=0.917


learning_rate=0.3: test accuracy=0.930


## 12. Failure Case: Overfitting with Boosting

Too many trees or too high learning rate overfits.


In [7]:
m_over = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.5, max_depth=10, random_state=42, eval_metric='logloss')
m_over.fit(X_train, y_train)
print(f"Overfit config: train={accuracy_score(y_train, m_over.predict(X_train)):.3f}, "
      f"test={accuracy_score(y_test, m_over.predict(X_test)):.3f}")
print("\nHigh train, lower test = overfitting.")


Overfit config: train=1.000, test=0.927

High train, lower test = overfitting.


## 13. Debugging: Common Errors

- **Overfitting**: too many trees, too deep.
- **Learning rate too high**: unstable.
- **Imbalanced data**: use scale_pos_weight.

## 14. Real-World Considerations

- Boosting is the default for tabular data.
- Use early stopping to avoid overfitting.
- XGBoost/LightGBM are faster and more flexible than sklearn's.

## 15. Common Mistakes

- Too many estimators without early stopping.
- Not tuning learning rate and depth together.

## 16. When NOT to Use

- When interpretability is critical (use a single tree or linear model).
- When data is tiny.

## 17. Challenge

Use early stopping with XGBoost and compare to a fixed number of trees.


In [8]:
# Challenge: early stopping
m_es = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.1, random_state=42, eval_metric='logloss', early_stopping_rounds=20)
m_es.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print(f"Early stopping best iteration: {m_es.best_iteration}")
print(f"Early stopping test accuracy:  {accuracy_score(y_test, m_es.predict(X_test)):.3f}")
print("\nEarly stopping finds the right number of trees automatically.")


Early stopping best iteration: 204
Early stopping test accuracy:  0.917

Early stopping finds the right number of trees automatically.


## 18. Closed-Book Recall

Without looking back:

1. What is the difference between bagging and boosting?
2. How does gradient boosting build trees sequentially?
3. What does the learning rate control?
4. Why is early stopping useful?

## 19. Teach-Back Questions

Explain to another person:

- How boosting corrects previous errors.
- The tradeoff between learning rate and number of trees.

## 20. Summary

You trained gradient boosting models with sklearn, XGBoost, and LightGBM, and compared them to random forests. Boosting is the state of the art for tabular data.

## 21. Further Experiment

- Tune max_depth and subsample.
- Try CatBoost.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, scikit-learn, xgboost, lightgbm
OUTPUTS: PASS
LAST VERIFIED: 2026-08-28
```
